# Dataset analysis

## 1. Setup and data loading

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONFIG_FILE = "qwen7b_lora_10.json"
# ──────────────────────────────────────────────────────────────────────────────

import json
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from datasets import load_dataset

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: "%.3f" % x)

config      = json.loads(Path("../configs/"+CONFIG_FILE).read_text(encoding="utf-8"))

OUTPUT_DIR   = Path(config["dataset_dir"])
DATASET_NAME = config["dataset_name"]
SEED         = config["dataset_seed"]

def load_and_transform(config, output_dir, dataset_name):
    full_path = output_dir / config["original_dataset"]
    if full_path.exists():
        print(f"Loading from cache: {full_path}")
        return pd.read_json(full_path)

    print("Downloading dataset...")
    df = load_dataset(dataset_name, split="train").to_pandas()
    df = df.drop(columns=config.get("drop_columns", []))
    for col in config.get("str_fillna_columns", []):
        df[col] = df[col].fillna("").astype(str)
    for col in config.get("int_to_str_columns", []):
        df[col] = df[col].apply(lambda x: "" if pd.isna(x) else str(int(x)))
    for col in config.get("str_to_int_columns", []):
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in config.get("str_to_float_columns", []):
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)
    for col in config.get("float_to_int_columns", []):
        df[col] = df[col].apply(lambda x: None if pd.isna(x) else int(x))
    return df

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df = load_and_transform(config, OUTPUT_DIR, DATASET_NAME)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 2. Dataset overview

In [ ]:
print("\n" + "=" * 80)
print("2. DATASET OVERVIEW")
print("=" * 80)

print("\n2.1 Column Information")
print("-" * 80)
print(f"\nTotal columns: {len(df.columns)}")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3d}. {col:<30} | Type: {str(df[col].dtype):<10} | Unique: {df[col].nunique():>8,}")

## 3. Data types and structure

In [ ]:
print("\n" + "=" * 80)
print("3. DATA TYPES AND STRUCTURE")
print("=" * 80)

print("\n3.1 Data Type Summary")
print("-" * 80)
dtype_summary = df.dtypes.value_counts().to_frame('Count')
dtype_summary['Percentage'] = (dtype_summary['Count'] / len(df.columns) * 100).round(2)
print(dtype_summary)

print("\n3.2 Detailed Data Types")
print("-" * 80)
df.info()

print("\n3.3 Memory Usage by Column")
print("-" * 80)
memory_usage = df.memory_usage(deep=True).to_frame('Bytes')
memory_usage['MB'] = (memory_usage['Bytes'] / 1024**2).round(3)
memory_usage = memory_usage.sort_values('Bytes', ascending=False).head(10)
print(memory_usage)

## 4. Data quality assessment

In [ ]:
print("\n" + "=" * 80)
print("4. DATA QUALITY ASSESSMENT")
print("=" * 80)

print("\n4.1 Missing Values Analysis")
print("-" * 80)
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2),
    'Data_Type': df.dtypes
})
missing_data = missing_data.sort_values('Missing_Percentage', ascending=False)
print(missing_data)

total_missing = df.isnull().sum().sum()
print(f"\nTotal missing values: {total_missing:,} ({total_missing / (len(df) * len(df.columns)) * 100:.2f}% of all values)")

print("\n4.2 Duplicate Records")
print("-" * 80)
duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates:,} ({duplicates / len(df) * 100:.2f}%)")

if duplicates > 0:
    duplicate_subset = df.duplicated(subset=None, keep=False).sum()
    print(f"Rows involved in duplicates: {duplicate_subset:,}")

## 5. Numerical features analysis

In [ ]:
print("\n" + "=" * 80)
print("5. NUMERICAL FEATURES ANALYSIS")
print("=" * 80)

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nFound {len(numerical_cols)} numerical columns")

print("\n5.1 Statistical Summary")
print("-" * 80)
print(df[numerical_cols].describe())

print("\n5.2 Extended Statistics")
print("-" * 80)
extended_stats = pd.DataFrame({
    'Column': numerical_cols,
    'Mean': [df[col].mean() for col in numerical_cols],
    'Median': [df[col].median() for col in numerical_cols],
    'Std': [df[col].std() for col in numerical_cols],
    'Skewness': [df[col].skew() for col in numerical_cols],
    'Kurtosis': [df[col].kurtosis() for col in numerical_cols],
    'CV': [df[col].std() / df[col].mean() if df[col].mean() != 0 else 0 for col in numerical_cols]
})
extended_stats = extended_stats.round(3)
print(extended_stats)

print("\n5.3 Distribution Analysis")
print("-" * 80)

n_cols_to_plot = min(len(numerical_cols), 12)
if n_cols_to_plot > 0:
    n_rows = (n_cols_to_plot + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
    axes = axes.flatten() if n_cols_to_plot > 1 else [axes]
    
    for idx, col in enumerate(numerical_cols[:n_cols_to_plot]):
        axes[idx].hist(df[col].dropna(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'{col}', fontweight='bold')
        axes[idx].set_xlabel('Value')
        axes[idx].set_ylabel('Frequency')
        axes[idx].grid(alpha=0.3)
        
        mean_val = df[col].mean()
        median_val = df[col].median()
        axes[idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
        axes[idx].axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.2f}')
        axes[idx].legend(fontsize=8)
    
    for idx in range(n_cols_to_plot, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

print("\n5.4 Outlier Detection")
print("-" * 80)

outlier_summary = []
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
    outlier_count = len(outliers)
    outlier_percentage = (outlier_count / len(df) * 100)
    
    outlier_summary.append({
        'Column': col,
        'Outlier_Count': outlier_count,
        'Outlier_Percentage': round(outlier_percentage, 2),
        'Lower_Bound': round(lower_bound, 3),
        'Upper_Bound': round(upper_bound, 3)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier_Percentage', ascending=False)
print(outlier_df)

n_cols_to_plot = min(len(numerical_cols), 12)
if n_cols_to_plot > 0:
    n_rows = (n_cols_to_plot + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
    axes = axes.flatten() if n_cols_to_plot > 1 else [axes]
    
    for idx, col in enumerate(numerical_cols[:n_cols_to_plot]):
        axes[idx].boxplot(df[col].dropna(), vert=True)
        axes[idx].set_title(f'{col}', fontweight='bold')
        axes[idx].set_ylabel('Value')
        axes[idx].grid(alpha=0.3)
    
    for idx in range(n_cols_to_plot, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

## 6. Categorical features analysis

In [ ]:
print("\n" + "=" * 80)
print("6. CATEGORICAL FEATURES ANALYSIS")
print("=" * 80)

categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"\nFound {len(categorical_cols)} categorical columns")

print("\n6.1 Categorical Features Summary")
print("-" * 80)
cat_summary = []
for col in categorical_cols:
    cat_summary.append({
        'Column': col,
        'Unique_Values': df[col].nunique(),
        'Most_Common': df[col].mode()[0] if len(df[col].mode()) > 0 else None,
        'Most_Common_Freq': df[col].value_counts().iloc[0] if len(df[col]) > 0 else 0,
        'Most_Common_Pct': round(df[col].value_counts(normalize=True).iloc[0] * 100, 2) if len(df[col]) > 0 else 0
    })
cat_summary_df = pd.DataFrame(cat_summary).sort_values('Unique_Values', ascending=False)
print(cat_summary_df)

print("\n6.2 Value Counts for Each Categorical Feature")
print("-" * 80)
for col in categorical_cols[:10]:
    print(f"\n{col}:")
    value_counts = df[col].value_counts().head(15)
    value_pcts = df[col].value_counts(normalize=True).head(15) * 100
    
    combined = pd.DataFrame({
        'Count': value_counts,
        'Percentage': value_pcts.round(2)
    })
    print(combined)

n_cols_to_plot = min(len(categorical_cols), 9)
if n_cols_to_plot > 0:
    n_rows = (n_cols_to_plot + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
    axes = axes.flatten() if n_cols_to_plot > 1 else [axes]
    
    for idx, col in enumerate(categorical_cols[:n_cols_to_plot]):
        top_values = df[col].value_counts().head(10)
        top_values.plot(kind='barh', ax=axes[idx], color='coral')
        axes[idx].set_title(f'Top 10 Values: {col}', fontweight='bold')
        axes[idx].set_xlabel('Count')
        axes[idx].invert_yaxis()
        axes[idx].grid(alpha=0.3)
    
    for idx in range(n_cols_to_plot, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

## 7. Correlation analysis

In [ ]:
print("\n" + "=" * 80)
print("7. CORRELATION ANALYSIS")
print("=" * 80)

if len(numerical_cols) > 1:
    print("\n7.1 Correlation Matrix")
    print("-" * 80)
    
    correlation_matrix = df[numerical_cols].corr()
    print(correlation_matrix)
    
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        correlation_matrix,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        ax=ax,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8}
    )
    ax.set_title("Correlation Heatmap", fontsize=16, fontweight="bold", pad=20)
    plt.tight_layout()
    plt.show()
    
    print("\n7.2 Highly Correlated Feature Pairs")
    print("-" * 80)
    
    corr_pairs = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i + 1, len(correlation_matrix.columns)):
            corr_pairs.append({
                'Feature_1': correlation_matrix.columns[i],
                'Feature_2': correlation_matrix.columns[j],
                'Correlation': correlation_matrix.iloc[i, j]
            })
    
    corr_df = pd.DataFrame(corr_pairs)
    corr_df['Abs_Correlation'] = corr_df['Correlation'].abs()
    corr_df = corr_df.sort_values('Abs_Correlation', ascending=False)
    
    high_corr = corr_df[corr_df['Abs_Correlation'] > 0.7].head(20)
    if len(high_corr) > 0:
        print("\nStrong correlations (|r| > 0.7):")
        print(high_corr[['Feature_1', 'Feature_2', 'Correlation']])
    else:
        print("\nNo strong correlations found (threshold: |r| > 0.7)")

## 8. Data distribution tests

In [ ]:
print("\n" + "=" * 80)
print("8. DATA DISTRIBUTION TESTS")
print("=" * 80)

print("\n8.1 Normality Tests (Shapiro-Wilk)")
print("-" * 80)

normality_results = []
for col in numerical_cols[:20]:
    sample_size = min(5000, len(df[col].dropna()))
    sample_data = df[col].dropna().sample(n=sample_size, random_state=42)
    
    if len(sample_data) >= 3:
        statistic, p_value = stats.shapiro(sample_data)
        is_normal = p_value > 0.05
        
        normality_results.append({
            'Column': col,
            'Statistic': round(statistic, 4),
            'P_Value': round(p_value, 6),
            'Is_Normal': 'Yes' if is_normal else 'No'
        })

normality_df = pd.DataFrame(normality_results)
print(normality_df)
print("\nNote: P-value > 0.05 suggests normal distribution")

## 9. Summary

In [ ]:
print("\n" + "=" * 80)
print("9. SUMMARY")
print("=" * 80)

print("\n9.1 Dataset Characteristics")
print("-" * 80)
print(f"Total Rows: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print(f"Numerical Columns: {len(numerical_cols)}")
print(f"Categorical Columns: {len(categorical_cols)}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n9.2 Data Quality Metrics")
print("-" * 80)
print(f"Total Missing Values: {df.isnull().sum().sum():,} ({df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100:.2f}%)")
print(f"Columns with Missing Values: {(df.isnull().sum() > 0).sum()}")
print(f"Duplicate Rows: {duplicates:,} ({duplicates / len(df) * 100:.2f}%)")

print("\n9.3 Feature Characteristics")
print("-" * 80)
if len(numerical_cols) > 0:
    print(f"Numerical Features Range: {df[numerical_cols].min().min():.3f} to {df[numerical_cols].max().max():.3f}")
    print(f"Average Skewness: {extended_stats['Skewness'].mean():.3f}")
    print(f"Average Kurtosis: {extended_stats['Kurtosis'].mean():.3f}")

if len(categorical_cols) > 0:
    print(f"Average Unique Categories: {cat_summary_df['Unique_Values'].mean():.0f}")
    print(f"Max Unique Categories: {cat_summary_df['Unique_Values'].max()}")

print("\n9.4 Correlation Insights")
print("-" * 80)
if len(numerical_cols) > 1:
    print(f"Strong Correlations (|r| > 0.7): {len(corr_df[corr_df['Abs_Correlation'] > 0.7])}")
    print(f"Moderate Correlations (0.5 < |r| < 0.7): {len(corr_df[(corr_df['Abs_Correlation'] > 0.5) & (corr_df['Abs_Correlation'] <= 0.7)])}")
    print(f"Weak Correlations (|r| < 0.3): {len(corr_df[corr_df['Abs_Correlation'] < 0.3])}")